# Notebook 03 — Tool use: letting the model call Python functions

**Time:** about 45 minutes

**What you will learn**
- What a *tool* is and why a model needs tools
- How the model **asks** for a tool, and how **your code** runs it
- The tool loop: ask → run tool → send result → answer
- Safety: only allowed tools, and a limit on steps

### The big idea
A language model only writes text. It does not know the current time and cannot read your product list.
So we give it **tools** — ordinary Python functions. The model replies *"please call `check_stock('SG101')`"*,
**our code** runs the function, and we send the result back. The model never runs anything itself.

⚠️ Tool calling needs a model that supports it. `llama3.2` does. Check any model with `ollama show <model>`
and look for **tools** under *Capabilities*.

### ▶️ Run the two setup cells below first
They check that Ollama is ready and create `ask()`, the one helper we use to talk to the model.

In [1]:
# STEP 0 — Is Ollama running? Is the model downloaded?
import ollama

MODEL = "llama3.2"     # local model (bigger one: "qwen2.5:7b")

try:
    installed = [m.model for m in ollama.list().models]
    print("Models on this computer:", installed)
    if MODEL in installed or MODEL + ":latest" in installed:
        print("✅ Ready to go with", MODEL)
    else:
        print(f"⚠️ {MODEL} is missing. Run: ollama pull {MODEL}")
except ConnectionError:
    print("⚠️ Ollama is not running. Start it, then run this again:")
    print("   Arch Linux: sudo systemctl start ollama")
    print("   Windows 11: open Ollama from the Start menu")

Models on this computer: ['llama3.2:latest']
✅ Ready to go with llama3.2


In [2]:
# STEP 1 — A helper: send a message, get the answer back as text
DEFAULT_ROLE = "You are a helpful assistant. Keep answers short."

def ask(prompt, system=DEFAULT_ROLE):
    reply = ollama.chat(
        model=MODEL,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": prompt}],
        options={"temperature": 0},    # 0 = more predictable
    )
    return reply.message.content

print("ask() is ready.")

ask() is ready.


## Part 1 — Write the tools (plain Python functions)

The Ollama library reads each function's **name**, **type hints** and **docstring** and turns them into a
description the model can understand. So clear docstrings matter!

In [9]:
# Tool 1: the time in a city
from datetime import datetime
from zoneinfo import ZoneInfo

CITY_ZONES = {"tokyo": "Asia/Tokyo", "london": "Europe/London",
              "karachi": "Asia/Karachi", "lahore": "Asia/Karachi",
              "new york": "America/New_York"}

def get_current_time(city: str) -> str:
    """Get the current local time in a city.

    Args:
        city: City name, for example Tokyo or London.
    """
    zone = CITY_ZONES.get(city.lower())
    if zone is None:
        return f"Unknown city '{city}'. Known: {list(CITY_ZONES)}"
    now = datetime.now(ZoneInfo(zone))
    return now.strftime("%H:%M on %A")

print(get_current_time("Tokyo"))

10:21 on Wednesday


In [ ]:
# Tool 2: compound interest
def calculate_interest(principal: float, rate_percent: float,
                       years: int) -> float:
    """Final amount after yearly compound interest.

    Args:
        principal: Starting amount of money.
        rate_percent: Yearly interest rate in percent, e.g. 5.
        years: Number of years.
    """
    growth = (1 + rate_percent / 100) ** years
    return round(principal * growth, 2)

print(calculate_interest(500, 5, 10))

In [ ]:
# Tool 3: search the product list
import pandas as pd
PRODUCTS = pd.read_csv("../datasets/products.csv")

def find_products(shape: str, max_price: float) -> list:
    """Find sunglasses of one frame shape up to a maximum price.
    Returns ids, names and prices (not the stock).

    Args:
        shape: Frame shape, for example round, square or oval.
        max_price: Highest price in US dollars.
    """
    is_sunglasses = PRODUCTS["category"] == "sunglasses"
    right_shape = PRODUCTS["shape"] == shape.lower()
    cheap_enough = PRODUCTS["price"] <= float(max_price)
    rows = PRODUCTS[is_sunglasses & right_shape & cheap_enough]
    return rows[["product_id", "name", "price"]].to_dict("records")

print(find_products("round", 100))

In [ ]:
# Tool 4: check the stock of one product
def check_stock(product_id: str) -> str:
    """Number of units in stock for one product.

    Args:
        product_id: Product id, for example SG101.
    """
    rows = PRODUCTS[PRODUCTS["product_id"] == product_id]
    if rows.empty:
        return f"Unknown product id '{product_id}'"
    units = int(rows.iloc[0]["stock"])
    return f"{units} units"

print(check_stock("SG101"))

We tested every tool **ourselves first**, without the model. Now the **allow-list**: the only tools the model may use.

In [ ]:
# The ALLOW-LIST: the only tools the model may use
TOOLS = {
    "get_current_time": get_current_time,
    "calculate_interest": calculate_interest,
    "find_products": find_products,
    "check_stock": check_stock,
}
print("Allowed tools:", list(TOOLS))

## Part 2 — One request: what does the model ask for?

We send a question **and the list of tools**. The model does not answer yet — it asks for a tool.

In [ ]:
question = "What time is it in Tokyo right now?"
reply = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": question}],
    tools=list(TOOLS.values()),       # give the model the tools
    options={"temperature": 0},
)

print("Text answer   :", repr(reply.message.content))
print("Tool requests :")
for call in reply.message.tool_calls or []:
    print("   ", call.function.name, call.function.arguments)

## Part 3 — Run a requested tool, safely

The model's request is only a *wish*. Our code decides:
- Is the tool on the allow-list? If not → refuse.
- Did the tool crash (for example, wrong argument types)? → send the error back as text, so the model can try again.

In [ ]:
def run_tool(call):
    name = call.function.name
    args = call.function.arguments
    if name not in TOOLS:
        return f"Error: '{name}' is not an allowed tool."
    try:
        return TOOLS[name](**args)
    except Exception as error:
        return f"Error while running {name}: {error}"

for call in reply.message.tool_calls or []:
    print(call.function.name, "→", run_tool(call))

## Part 4 — The complete tool loop (the heart of every agent)

```
question → model → wants a tool? ──yes──► our code runs it ──► result back to the model ──┐
                       │                                                                    │
                       no ◄─────────────────────────────────────────────────────────────────┘
                       ▼
                 final answer
```
`max_steps` stops the loop if the model keeps asking for tools forever.

In [ ]:
SYSTEM = ("You are a helpful shop and time assistant. "
          "Use the tools to find facts. Never guess prices, "
          "stock or times. If no tool is needed, just answer.")

def agent(question, max_steps=5):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": question}]
    for step in range(max_steps):
        reply = ollama.chat(model=MODEL, messages=messages,
                            tools=list(TOOLS.values()),
                            options={"temperature": 0})
        messages.append(reply.message)      # remember the reply
        if not reply.message.tool_calls:    # no tool → final answer
            return reply.message.content
        for call in reply.message.tool_calls:
            name = call.function.name
            args = call.function.arguments
            result = run_tool(call)
            print(f"   🔧 {name}({args}) → {result}")
            messages.append({"role": "tool", "tool_name": name,
                             "content": str(result)})
    return "Stopped: too many steps."

In [ ]:
print(agent("What time is it in Tokyo right now?"))

In [ ]:
question = ("If I invest $500 at 5% per year for 10 years, "
            "how much will I have?")
print(agent(question))

### A question that needs two tools
To answer this, the model must first **find** round sunglasses, then **check the stock** of each one.

In [ ]:
question = "Which round sunglasses under $100 are in stock?"
print(agent(question))

### Check the agent's answer against the truth
We can compute the correct answer directly with pandas. **Always check an agent against known answers.**

In [ ]:
truth = PRODUCTS[(PRODUCTS["shape"] == "round") &
                 (PRODUCTS["category"] == "sunglasses") &
                 (PRODUCTS["price"] < 100) &
                 (PRODUCTS["stock"] > 0)]
print(truth[["product_id", "name", "price", "stock"]])

### A question that needs no tool
A good agent does **not** call tools when it does not need them.

In [ ]:
print(agent("What is a time zone? Answer in one sentence."))

## 💬 Discuss
- Did the agent call the right tools? Did it ever call a tool it did not need?
- Small models sometimes send wrong arguments. Where in our code is that caught?
- Why must **our code**, not the model, run the tools?

## ✍️ Try it yourself
1. Ask the agent: *"What time is it in Lahore?"* and then *"What time is it in Atlantis?"*
2. Write a new tool `get_price(product_id: str) -> float` (with a docstring!), add it to `TOOLS`, and ask
   *"How much does SG104 cost?"*

## ✅ Summary
- A **tool** is a Python function the model can ask for; our code runs it.
- The **tool loop**: model asks → code runs → result goes back → model answers.
- Safety: an **allow-list** of tools, errors sent back as text, and a **step limit**.

**Next:** Notebook 04 — measuring how good an AI system really is (**evaluations**).